# AXIFEM element evidence

This result-bearing notebook records the current proof that the six production AXIFEM element paths run through the NGSolve `.vol` route and are guarded by executable regression gates: P1, Q1, P2, Q2, P2 curved, and Q2 curved.


In [1]:
from pathlib import Path
import json

relative = Path("validation_test/axifem/axifem_element_evidence.json")
for root in [Path.cwd(), *Path.cwd().parents]:
    candidate = root / relative
    if candidate.exists():
        evidence = json.loads(candidate.read_text(encoding="utf-8"))
        break
else:
    raise FileNotFoundError(relative)

print(json.dumps(evidence["version_stamp"], indent=2, ensure_ascii=False))


{
  "runtime_radia_version": "4.95.5",
  "executed_at_utc": "2026-07-05T02:09:01.637577+00:00",
  "python": "3.12.10",
  "python_executable": "C:\\Program Files\\Python312\\python.exe",
  "platform": "Windows-2022Server-10.0.20348-SP0",
  "git_head": "19e54c13",
  "git_dirty": true,
  "mesh_route": "Netgen .vol -> ngsolve.Mesh(path)",
  "pytest_passed": 34
}


## Evidence matrix

Each row links one element path to the `.vol` production selection API and the regression gate that would fail if the path regressed.


In [2]:
from IPython.display import Markdown, display
rows = evidence["evidence_matrix"]
headers = ["Element path", "Production selection", "Evidence gate", "What would fail if broken"]
lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
for row in rows:
    lines.append("| " + " | ".join(str(row[h]).replace("|", "\\|") for h in headers) + " |")
display(Markdown("\n".join(lines)))


| Element path | Production selection | Evidence gate | What would fail if broken |
| --- | --- | --- | --- |
| P1 triangle | H1Henrotte(mesh, order=1) on triangles loaded through Netgen .vol | test_de_rham_identities.py + test_element_matrices.py + test_python_reference_consistency.py + test_axifem_guards.py | wrong ET_TRIG vertex convention, singular (r^2,z) Vandermonde, Python FEMM-reference drift, or loss of constant-mode/null-mode checks |
| Q1 quad | H1Henrotte(mesh, order=1) on axis-aligned quads loaded through Netgen .vol | test_de_rham_identities.py + test_q1_vdof.py + test_axifem_guards.py + V-DOF reference in test_python_reference_consistency.py | uniform B_z field no longer reproduced, non-axis-aligned quads silently accepted, or V-DOF stiffness/mass spectrum drifts |
| P2 triangle | H1Henrotte(mesh, order=2) on triangles loaded through Netgen .vol | test_de_rham_identities.py + test_p2_axis_eddy.py + test_python_reference_consistency.py | dead face-center DOFs become free, K/M lose full rank, or P1/P2 slowest eddy mode diverges |
| Q2 quad | H1Henrotte(mesh, order=2, curvedquad=False) on axis-aligned quads loaded through Netgen .vol | test_de_rham_identities.py + test_q2_curved.py straight-quad equivalence + test_axifem_guards.py + Cauer Q2 stored validation summary | axis-aligned Q2 eigenvalue changes, non-axis-aligned closed-form input is not rejected, or Q2 Cauer time constants drift |
| P2 curved triangle | mesh.Curve(2), saved as .vol, reloaded with ngsolve.Mesh(path), then H1Henrotte(mesh, order=2) | test_de_rham_identities.py + test_p2_curved_magsta.py | Curve(2) no longer improves curved-boundary volume/total-flux error or curved P2 identities drift |
| Q2 curved quad | H1Henrotte(mesh, order=2, curvedquad=True) on .vol-backed mapped/skewed quads | test_de_rham_identities.py + test_q2_curved.py + test_axifem_guards.py | curved Q2 no longer matches straight Q2 on rectangles, no longer converges on annular skewed quads, or cannot accept skewed .vol quads |

## Pytest proof

The pytest artifact intentionally excludes this notebook's self-check test to avoid circular evidence. The full `tests/axifem` suite is run separately in CI/local verification.


In [3]:
print(" ".join(evidence["pytest"]["command"]))
print()
print(evidence["pytest"]["stdout"])


python -m pytest tests\axifem -q --ignore=tests\axifem\test_docs_notebook_evidence.py

============================= test session starts =============================
platform win32 -- Python 3.12.10, pytest-9.0.3, pluggy-1.6.0
rootdir: W:\00_CAE\Radia\01_GitHub
configfile: pytest.ini (WARNING: ignoring pytest config in pyproject.toml!)
plugins: anyio-4.13.0, rerunfailures-16.3
collected 34 items

tests\axifem\test_axifem_guards.py ....                                  [ 11%]
tests\axifem\test_de_rham_identities.py .....                            [ 26%]
tests\axifem\test_element_matrices.py ......                             [ 44%]
tests\axifem\test_heat_axisym_e2e.py .                                   [ 47%]
tests\axifem\test_heat_henrotte_smoke.py ....                            [ 58%]
tests\axifem\test_p2_axis_eddy.py ..                                     [ 64%]
tests\axifem\test_p2_curved_magsta.py ..                                 [ 70%]
tests\axifem\test_python_reference_cons

## Cross-cutting gates

These gates capture the production assumptions that cut across individual element families.


In [4]:
from IPython.display import Markdown, display
rows = evidence["cross_cutting_gates"]
headers = ["gate", "files", "contract"]
lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
for row in rows:
    lines.append("| " + " | ".join(str(row[h]).replace("|", "\\|") for h in headers) + " |")
display(Markdown("\n".join(lines)))


| gate | files | contract |
| --- | --- | --- |
| .vol-backed_mesh_route | tests/axifem/_vol_mesh.py, test_de_rham_identities.py, test_axifem_guards.py, test_heat_henrotte_smoke.py | Tests save Netgen meshes to .vol and reload with ngsolve.Mesh(path), matching the production entry route. |
| du_rham_identity | tests/axifem/test_de_rham_identities.py | Limited Henrotte scalar lane identity: u=1, r^2, z interpolate exactly and satisfy B_r=-du/dz, B_z=du/dr+u/r. |
| closed_form_quad_guard | src/ext/axifem/axi_henrotte_fespace.cpp, tests/axifem/test_axifem_guards.py | Default Q1/Q2 closed-form quads reject non-axis-aligned .vol quads; curvedquad=True is the opt-in general-quad path. |
| boundary_value_trace | tests/axifem/test_de_rham_identities.py | Axis-aligned boundary value traces integrate against analytic edge integrals after .vol reload. |

## Stored validation summaries

The Cauer disk summaries are stored regression references. They are kept with the evidence but are not live external solver calls.


In [5]:
from IPython.display import Markdown, display
headers = ["family", "selected_mesh", "mesh", "tau_pair_us_first6", "stored_bem_reference_us_first6", "first_tau_relative_to_reference"]
rows = []
for row in evidence["cauer_validation_summary"]:
    mesh = row.get("mesh", {})
    item = dict(row)
    item["mesh"] = f"ne={mesh.get('ne')}, ndof={mesh.get('ndof')}, free={mesh.get('free')}"
    rows.append(item)
if rows:
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for row in rows:
        lines.append("| " + " | ".join(str(row[h]).replace("|", "\\|") for h in headers) + " |")
    display(Markdown("\n".join(lines)))
else:
    display(Markdown("No stored Cauer summaries were found."))


| family | selected_mesh | mesh | tau_pair_us_first6 | stored_bem_reference_us_first6 | first_tau_relative_to_reference |
| --- | --- | --- | --- | --- | --- |
| Q1 quad | very fine | ne=15170, ndof=15438, free=14904 | [218.052147, 77.773176, 39.374895, 23.140683, 16.064211, 13.011787] | [224.307059, 88.423956, 49.539868, 32.090204, 23.339598, 22.588437] | 0.027885 |
| Q2 quad | fine | ne=2530, ndof=10323, free=9919 | [218.707571, 78.121442, 39.54416, 23.157879, 16.073587, 13.118726] | [224.307059, 88.423956, 49.539868, 32.090204, 23.339598, 22.588437] | 0.024963 |

## Timing

The result JSON keeps a small timing breakdown. Heavy phases are listed first.


In [6]:
print(json.dumps(evidence["timing_breakdown_s"], indent=2, ensure_ascii=False))


{
  "pytest_axifem_vol_backed_gates": 2.883,
  "cauer_json_summary_load": 0.0,
  "notebook_artifact_generation": 0.0
}
